# RAG Pipeline Exercise — SOLUTION

This notebook contains the complete solution with all TODOs filled in and discussion answers provided.

## What is RAG?

Large Language Models (like GPT-4) are powerful, but they can only answer from what they memorized during training. If you ask about something they haven't seen, they might **hallucinate** — confidently make up a wrong answer.

**Retrieval-Augmented Generation (RAG)** solves this by adding a search step:

1. **Search** a knowledge base for passages relevant to the user's question
2. **Inject** those passages into the LLM's prompt as context
3. **Ask** the LLM to answer based on what it just read

```
Documents → Chunk → Embed → Store in Vector DB
                                    ↓
User Query → Embed Query → Retrieve Top-k → Augment Prompt → LLM → Answer
```

---
## Setup

In [ ]:
# !pip install datasets langchain langchain-openai langchain-community chromadb openai tiktoken matplotlib numpy

In [ ]:
import os
import helper
from helper import format_docs

from datasets import load_dataset
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# os.environ["OPENAI_API_KEY"] = "sk-..."
assert os.environ.get("OPENAI_API_KEY"), "Set your OPENAI_API_KEY before continuing."

---
## Task 1 — Data Loading & Exploration

We load the SQuAD v2 dataset from HuggingFace. It contains Wikipedia paragraphs paired with questions and answers. We extract and deduplicate the paragraphs to get our knowledge base.

In [ ]:
squad = load_dataset("rajpurkar/squad_v2", split="validation")
print(f"Total examples in validation set: {len(squad)}")

In [ ]:
all_documents = list(dict.fromkeys(squad["context"]))
print(f"Unique documents: {len(all_documents)}")

documents = all_documents[:200]
print(f"Working set: {len(documents)} documents")
print(f"\nHere's the first document (first 300 chars):")
print(f"{documents[0][:300]}...")

In [ ]:
helper.plot_document_length_distribution(documents)

### Discussion

The documents vary significantly in length, ranging from a few hundred to over a thousand characters. Most documents fall in the 400–800 character range. This variation is typical of Wikipedia paragraphs: some topics are covered briefly, others in more detail.

This matters for chunking because documents longer than our chunk size (500 characters) will be split into multiple chunks, while shorter documents will become a single chunk. The uneven distribution means some topics will have more chunks (and thus more chances to be retrieved) than others.

---
## Task 2 — Document Chunking

We split documents into smaller chunks because embedding models work best with shorter texts, and retrieval is more precise when each chunk covers a single coherent topic.

In [ ]:
# SOLUTION: Create the text splitter and split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.create_documents(documents)

print(f"We split {len(documents)} documents into {len(chunks)} chunks.")
print(f"\nHere's what the first chunk looks like:")
print(f"{chunks[0].page_content[:200]}...")

In [ ]:
helper.plot_chunk_length_distribution(chunks)

### Discussion

If chunks were **very small** (e.g., 50 characters), each chunk would be just a sentence fragment. This would make retrieval unreliable because a short fragment doesn't carry enough meaning to match well against a question. The embedding of "in France during" isn't very useful for finding answers about French history.

If chunks were **very large** (e.g., 5000 characters), each chunk would contain multiple topics. The embedding would average across all of them, making it harder to match a specific question. Also, we'd be sending a lot of irrelevant text to the LLM, wasting tokens and potentially confusing it.

The **chunk_overlap=50** parameter means that the end of one chunk and the beginning of the next share 50 characters. This prevents a situation where an important sentence is split exactly at its midpoint between two chunks, making it unreadable in both. The overlap ensures continuity.

---
## Task 3 — Embeddings & Vector Store

We convert each chunk into a numerical vector (embedding) and store all vectors in ChromaDB. When we search later, ChromaDB will find the chunks whose vectors are most similar to the query.

In [ ]:
# SOLUTION: Create embedding model and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready — {vectorstore._collection.count()} vectors indexed.")

In [ ]:
test_queries = [
    "Who was the first president of the United States?",
    "What is photosynthesis?",
    "What is the best pizza topping?",  # out-of-domain!
]

for query in test_queries:
    results = vectorstore.similarity_search_with_score(query, k=3)
    helper.display_retrieval_results(query, results)

### Discussion

The in-domain queries retrieve relevant Wikipedia passages. The similarity scores are relatively low (ChromaDB uses distance, so lower = more similar), indicating strong matches. The retrieved passages contain information that could help answer the questions.

The out-of-domain query about pizza toppings still retrieves *something* — the vector store always returns results because it finds the "least dissimilar" vectors. However, the passages are clearly unrelated to pizza, and the distance scores are higher, indicating weaker matches. This is an important observation: a vector store can't tell you "I have nothing relevant" — it always returns the closest matches, even if they're not very close. This is why our prompt template needs to instruct the LLM to say "I don't know" when the context isn't helpful.

---
## Task 4 — Build the RAG Chain

We connect the retriever, prompt, LLM, and output parser into a single chain. The key creative decision is the **prompt template** — it controls how the LLM uses the retrieved context.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# SOLUTION: The prompt template
RAG_PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

Rules:
- Answer ONLY based on the context below. Do not use any prior knowledge.
- If the context does not contain enough information to answer the question, say "I don't know".
- Keep your answer concise and to the point.

Context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
test_questions = [
    "Who was the first president of the United States?",
    "What is photosynthesis?",
    "What is the best pizza topping?",  # out-of-domain
]

for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

### Discussion

The RAG chain answers in-domain questions by synthesizing information from the retrieved Wikipedia passages. For the out-of-domain pizza question, it correctly says "I don't know" because the prompt explicitly instructs it to only answer from the context.

Without the "I don't know" instruction, the LLM would likely answer the pizza question from its own knowledge (e.g., "Many people enjoy mozzarella..."), which defeats the purpose of RAG. The prompt is our main tool for controlling the LLM's behavior — even small changes to the wording can significantly affect the output.

---
## Task 5 — Evaluation & Comparison

We evaluate the pipeline systematically and compare it against a naive LLM (no retrieval) to understand when RAG actually helps.

In [ ]:
eval_set = helper.build_eval_set(squad, documents, n=30)
print(f"\nEvaluation set: {len(eval_set)} questions")
print(f"\nExample:")
print(f"  Question: {eval_set[0]['question']}")
print(f"  Answer:   {eval_set[0]['answer']}")

In [ ]:
eval_questions = [ex["question"] for ex in eval_set[:10]]
gold_answers = [ex["answer"] for ex in eval_set[:10]]

print("Running RAG chain on evaluation questions...\n")
rag_answers = []
for i, q in enumerate(eval_questions):
    answer = rag_chain.invoke(q)
    rag_answers.append(answer)
    print(f"[{i+1}/10] Q: {q}")
    print(f"       A: {answer}")
    print(f"    Gold: {gold_answers[i]}\n")

In [ ]:
# SOLUTION: Build the naive chain
naive_prompt = ChatPromptTemplate.from_template(
    "Answer the following question concisely:\n\n{question}"
)
naive_chain = naive_prompt | llm | StrOutputParser()

In [ ]:
print("Running naive chain (no retrieval) on the same questions...\n")
naive_answers = []
for i, q in enumerate(eval_questions):
    answer = naive_chain.invoke(q)
    naive_answers.append(answer)
    print(f"[{i+1}/10] Q: {q}")
    print(f"       A: {answer}\n")

In [ ]:
helper.display_comparison_table(eval_questions, rag_answers, naive_answers, gold_answers)

In [ ]:
# SOLUTION: Compute substring accuracy
rag_accuracy = sum(
    1 for ra, ga in zip(rag_answers, gold_answers)
    if ga.lower() in ra.lower()
) / len(eval_questions)

naive_accuracy = sum(
    1 for na, ga in zip(naive_answers, gold_answers)
    if ga.lower() in na.lower()
) / len(eval_questions)

print(f"RAG accuracy:   {rag_accuracy:.1%}")
print(f"Naive accuracy: {naive_accuracy:.1%}")

In [ ]:
helper.plot_evaluation_results({
    "RAG Accuracy": rag_accuracy,
    "Naive Accuracy": naive_accuracy,
})

### Discussion

RAG generally outperforms the naive LLM, especially on questions about **specific factual details** (dates, names, numbers, specific events) that appear in the Wikipedia passages. These are exactly the kind of details an LLM might not have memorized or might get wrong.

The naive LLM can still answer **well-known general knowledge questions** correctly (e.g., "Who was the first president?") because this information was heavily represented in its training data. For these questions, RAG doesn't add much value.

There are cases where RAG might perform *worse* than the naive chain:
- If the retriever returns **irrelevant chunks**, the LLM might get confused or say "I don't know" even though it could answer from memory.
- If the **gold answer phrasing** doesn't match the LLM's phrasing (e.g., gold says "1066" but the LLM says "in the year 1066 AD"), substring matching might give a false negative.

In real-world applications, RAG is essential when you need the system to answer from **private or recent data** — company documents, legal contracts, updated manuals — that the LLM has never seen during training.